# 🔐 Multimodal Biometric Fusion — Face + Voice

---

**Goal:** Identify a person using TWO modalities — face photo + voice clip.

```
Late Fusion:
  Photo → [ArcFace]         → face_embedding (512-dim)  → face_score
  Audio → [Speaker Encoder]  → voice_embedding (256-dim) → voice_score

  Fused score = w1 × face_score + w2 × voice_score
```

**This is real biometric fusion** — same idea behind Aadhaar (face + iris + fingerprint).

**Data needed:** 3 players × 3 samples each = 9 face photos + 9 audio clips

```
face_images/   dhoni_a.jpg, dhoni_b.jpg, dhoni_c.jpg
               kholi_a.jpg, kholi_b.jpg, kholi_c.jpg
               sachin_a.jpg, sachin_b.jpg, sachin_c.jpg

voice_clips/   dhoni_a.wav, dhoni_b.wav, dhoni_c.wav
               kholi_a.wav, kholi_b.wav, kholi_c.wav
               sachin_a.wav, sachin_b.wav, sachin_c.wav
```

**Platform:** Google Colab (GPU recommended)

---
## 0. Setup

In [ ]:
!git clone https://github.com/deepanrajm/GL_MML.git

In [1]:
!pip install insightface onnxruntime-gpu resemblyzer matplotlib seaborn librosa -q
# If GPU onnxruntime fails, use: pip install onnxruntime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.5/439.5 kB 18.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 8.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 9.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.8/252.8 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 97.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 93.5 MB/s eta 0:00:00


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import librosa
from PIL import Image
import os
import warnings
warnings.filterwarnings('ignore')

def cosine_sim(a, b):
    """Cosine similarity — same math as CLIP!"""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def get_person(name):
    """Extract person name from filename: 'dhoni_a' → 'dhoni'"""
    return name.rsplit('_', 1)[0]

print("✅ Ready!")

✅ Ready!


In [3]:
# Load models
from insightface.app import FaceAnalysis
from resemblyzer import VoiceEncoder, preprocess_wav

# Face model
print("Loading InsightFace (ArcFace)...")
face_app = FaceAnalysis(name='buffalo_l', providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
face_app.prepare(ctx_id=0, det_size=(640, 640))
print("✅ Face model loaded! (ArcFace → 512-dim embedding)")

# Voice model
print("Loading Resemblyzer (Speaker Encoder)...")
voice_encoder = VoiceEncoder()
print("✅ Voice model loaded! (d-vector → 256-dim embedding)")

print("\n💡 Both models do the same thing:")
print("   ArcFace:     face photo → 512-dim vector → cosine similarity")
print("   Resemblyzer: audio clip → 256-dim vector → cosine similarity")
print("   Same concept as CLIP — just different modalities!")

Loading InsightFace (ArcFace)...
download_path: /root/.insightface/models/buffalo_l


100%|██████████| 281857/281857 [00:02<00:00, 101968.31KB/s]


Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'sdpa_kernel': '0', 'use_tf32': '1', 'fuse_conv_bias': '0', 'prefer_nhwc': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_enable': '0', 'use_ep_level_unified_stream': '0', 'device_id': '0', 'has_user_compute_stream': '0', 'gpu_external_empty_cache': '0', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'cudnn_conv1d_pad_to_nc1d': '0', 'gpu_mem_limit': '18446744073709551615', 'gpu_external_alloc': '0', 'gpu_external_free': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'do_copy_in_default_stream': '1', 'enable_cuda_graph': '0', 'user_compute_stream': '0', 'cudnn_conv_use_max_workspace': '1'}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with o

In [ ]:


# 🔄 CHANGE these paths to your folders
face_folder = "GL_MML/Unit_4/face_images"
audio_folder = "GL_MML/Unit_4/voices_for_images"

print("Face images:")
for f in sorted(os.listdir(face_folder)):
    if f.lower().endswith(('.jpg', '.jpeg', '.png')):
        print(f"  ✅ {f}")

print("\nAudio clips:")
for f in sorted(os.listdir(audio_folder)):
    if f.lower().endswith(('.wav', '.mp3', '.m4a', '.flac')):
        print(f"  ✅ {f}")

---
---
# Step 1: Extract Face Embeddings

In [ ]:
face_data = {}    # {name: embedding}
face_images = {}  # {name: image}

print("═" * 70)
print("Extracting Face Embeddings (ArcFace → 512-dim)")
print("═" * 70)

for f in sorted(os.listdir(face_folder)):
    if f.lower().endswith(('.jpg', '.jpeg', '.png')):
        name = os.path.splitext(f)[0]
        img = cv2.imread(os.path.join(face_folder, f))
        face_images[name] = img

        faces = face_app.get(img)
        if len(faces) > 0:
            face_data[name] = faces[0].embedding
            print(f"  ✅ {name}: {faces[0].embedding.shape[0]}-dim embedding")
        else:
            print(f"  ❌ {name}: no face detected!")

print(f"\n{len(face_data)} face embeddings extracted!")

# Show detected faces
n = len(face_data)
cols = min(n, 5)
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(3.5*cols, 3.5*rows))
axes_flat = [axes] if n == 1 else (list(axes) if rows == 1 else [ax for row in axes for ax in row])

for idx, (name, img) in enumerate(face_images.items()):
    if name in face_data:
        axes_flat[idx].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes_flat[idx].set_title(f"{name}\n({get_person(name)})", fontsize=9)
        axes_flat[idx].axis('off')
for idx in range(len(face_data), len(axes_flat)):
    axes_flat[idx].axis('off')
plt.suptitle("Face Images", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
# Step 2: Extract Voice Embeddings

In [ ]:
voice_data = {}  # {name: embedding}

print("═" * 70)
print("Extracting Voice Embeddings (Resemblyzer → 256-dim)")
print("═" * 70)

for f in sorted(os.listdir(audio_folder)):
    if f.lower().endswith(('.wav', '.mp3', '.m4a', '.flac')):
        name = os.path.splitext(f)[0]
        filepath = os.path.join(audio_folder, f)

        try:
            wav, sr = librosa.load(filepath, sr=16000)
            wav = preprocess_wav(wav)
            embedding = voice_encoder.embed_utterance(wav)
            voice_data[name] = embedding
            duration = len(wav) / 16000
            print(f"  ✅ {name}: {embedding.shape[0]}-dim embedding ({duration:.1f}s audio)")
        except Exception as e:
            print(f"  ❌ {name}: error — {e}")

print(f"\n{len(voice_data)} voice embeddings extracted!")

# Find common names
common_names = sorted(set(face_data.keys()) & set(voice_data.keys()))
persons = sorted(set(get_person(n) for n in common_names))

print(f"\nCommon samples (have both face + voice): {len(common_names)}")
for p in persons:
    samples = [n for n in common_names if get_person(n) == p]
    print(f"  {p}: {samples}")

print(f"\n💡 Face: {list(face_data.values())[0].shape[0]}-dim  |  Voice: {list(voice_data.values())[0].shape[0]}-dim")
print(f"   Different dimensions, different modalities, but SAME similarity concept!")

---
# Step 3: Face Similarity Matrix

In [ ]:
n = len(common_names)
face_matrix = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        face_matrix[i][j] = cosine_sim(face_data[common_names[i]], face_data[common_names[j]])

plt.figure(figsize=(10, 8))
sns.heatmap(face_matrix, xticklabels=common_names, yticklabels=common_names,
            annot=True, fmt='.2f', cmap='RdYlGn', vmin=0, vmax=1,
            linewidths=0.5, linecolor='white')
plt.title('MODALITY 1: Face Similarity (ArcFace)\nSame person → GREEN, Different → RED',
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("💡 Look for 3×3 GREEN blocks along the diagonal.")
print("   Each block = same person's photos matching each other.")
print("   Off-diagonal = different people → should be RED (low similarity).")

---
# Step 4: Voice Similarity Matrix

In [ ]:
voice_matrix = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        voice_matrix[i][j] = cosine_sim(voice_data[common_names[i]], voice_data[common_names[j]])

plt.figure(figsize=(10, 8))
sns.heatmap(voice_matrix, xticklabels=common_names, yticklabels=common_names,
            annot=True, fmt='.2f', cmap='RdYlGn', vmin=0, vmax=1,
            linewidths=0.5, linecolor='white')
plt.title('MODALITY 2: Voice Similarity (Speaker Encoder)\nSame speaker → GREEN, Different → RED',
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("💡 Compare with the face heatmap above.")
print("   Some pairs might be confused by face but clear by voice, or vice versa.")
print("   That's exactly WHY we fuse them!")

---
# Step 5: Late Fusion — Face + Voice Combined

Three heatmaps side by side: **Face only | Voice only | Fused**

In [ ]:
w_face, w_voice = 0.6, 0.4
fused_matrix = w_face * face_matrix + w_voice * voice_matrix

fig, axes = plt.subplots(1, 3, figsize=(24, 7))

sns.heatmap(face_matrix, xticklabels=common_names, yticklabels=common_names,
            annot=True, fmt='.2f', cmap='RdYlGn', vmin=0, vmax=1,
            linewidths=0.5, linecolor='white', ax=axes[0])
axes[0].set_title('Face Only (ArcFace)', fontsize=12, fontweight='bold')

sns.heatmap(voice_matrix, xticklabels=common_names, yticklabels=common_names,
            annot=True, fmt='.2f', cmap='RdYlGn', vmin=0, vmax=1,
            linewidths=0.5, linecolor='white', ax=axes[1])
axes[1].set_title('Voice Only (Speaker Encoder)', fontsize=12, fontweight='bold')

sns.heatmap(fused_matrix, xticklabels=common_names, yticklabels=common_names,
            annot=True, fmt='.2f', cmap='RdYlGn', vmin=0, vmax=1,
            linewidths=0.5, linecolor='white', ax=axes[2])
axes[2].set_title(f'FUSED ({w_face:.0%} Face + {w_voice:.0%} Voice)', fontsize=12, fontweight='bold')

plt.suptitle('Late Fusion: Face + Voice → Combined Biometric Score',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print("💡 The FUSED heatmap should be CLEANER than either individual one.")
print("   Same-person blocks should be more uniformly GREEN.")
print("   Different-person cells should be more uniformly RED.")

---
# Step 6: Experiment with Fusion Weights

In [ ]:
# Try different face/voice weight ratios and compute accuracy
weights = np.linspace(0, 1, 21)  # face weight from 0.0 to 1.0
threshold = 0.5

accuracy_per_weight = []

for w in weights:
    correct = 0
    total = 0

    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            fused_score = w * face_matrix[i][j] + (1 - w) * voice_matrix[i][j]
            same_person = get_person(common_names[i]) == get_person(common_names[j])
            predicted_match = fused_score > threshold

            if same_person == predicted_match:
                correct += 1
            total += 1

    accuracy_per_weight.append(correct / total if total > 0 else 0)

# Plot
plt.figure(figsize=(10, 6))
plt.plot(weights, accuracy_per_weight, 'b-o', linewidth=2, markersize=6)
plt.xlabel('Face Weight (w₁)\nVoice Weight = 1 - w₁', fontsize=12)
plt.ylabel('Identification Accuracy', fontsize=12)
plt.title('How Fusion Weights Affect Accuracy', fontsize=13, fontweight='bold')

plt.axvline(x=0, color='orange', linestyle='--', alpha=0.7, label='Voice only (w=0)')
plt.axvline(x=1, color='green', linestyle='--', alpha=0.7, label='Face only (w=1)')

best_w = weights[np.argmax(accuracy_per_weight)]
best_acc = max(accuracy_per_weight)
plt.axvline(x=best_w, color='red', linestyle='-', alpha=0.7, label=f'Best: w={best_w:.2f}')
plt.scatter([best_w], [best_acc], color='red', s=150, zorder=5)

plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.ylim(-0.05, 1.05)
plt.tight_layout()
plt.show()

print(f"\n  Voice only (w=0.0):     {accuracy_per_weight[0]:.1%}")
print(f"  Face only  (w=1.0):     {accuracy_per_weight[-1]:.1%}")
print(f"  Best fusion (w={best_w:.2f}):   {best_acc:.1%}")
print(f"\n💡 If fused accuracy > either modality alone → fusion HELPS!")
print(f"   This is WHY multimodal biometric systems exist.")

---
# Step 7: Robustness — When One Modality Fails

In [ ]:
# Pick two samples of the SAME person
same_pair = None
for i in range(n):
    for j in range(i+1, n):
        if get_person(common_names[i]) == get_person(common_names[j]):
            same_pair = (common_names[i], common_names[j])
            break
    if same_pair:
        break

if same_pair:
    p1, p2 = same_pair
    person = get_person(p1)

    real_face = cosine_sim(face_data[p1], face_data[p2])
    real_voice = cosine_sim(voice_data[p1], voice_data[p2])
    real_fused = 0.6 * real_face + 0.4 * real_voice

    print("═" * 70)
    print("Robustness Test: What if one modality fails?")
    print("═" * 70)

    print(f"\n  Same person: {p1} vs {p2} ({person})")
    print(f"\n  --- Normal case (both modalities work) ---")
    print(f"  Face: {real_face:.4f}  Voice: {real_voice:.4f}  Fused: {real_fused:.4f} ✅")

    # Scenario 1: Face fails (sunglasses, bad photo, dark image)
    bad_face = 0.15
    print(f"\n  --- Scenario 1: Face fails (sunglasses, bad angle) ---")
    print(f"  Face only:  {bad_face:.4f}              ← FAILS (below 0.4 threshold)")
    print(f"  Voice only: {real_voice:.4f}              ← still works")
    print(f"  Fused:      {0.6*bad_face + 0.4*real_voice:.4f}              ← RECOVERED by voice!")

    # Scenario 2: Voice fails (noisy room, short clip, background music)
    bad_voice = 0.15
    print(f"\n  --- Scenario 2: Voice fails (noisy room, bad audio) ---")
    print(f"  Face only:  {real_face:.4f}              ← still works")
    print(f"  Voice only: {bad_voice:.4f}              ← FAILS (below threshold)")
    print(f"  Fused:      {0.6*real_face + 0.4*bad_voice:.4f}              ← RECOVERED by face!")

    # Scenario 3: Both fail
    print(f"\n  --- Scenario 3: Both modalities fail ---")
    print(f"  Fused:      {0.6*bad_face + 0.4*bad_voice:.4f}              ← both fail → fusion also fails")

    # Visual comparison
    scenarios = ['Normal', 'Face Fails', 'Voice Fails', 'Both Fail']
    face_scores = [real_face, bad_face, real_face, bad_face]
    voice_scores = [real_voice, real_voice, bad_voice, bad_voice]
    fused_scores = [0.6*f + 0.4*v for f, v in zip(face_scores, voice_scores)]

    x = np.arange(len(scenarios))
    width = 0.25

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x - width, face_scores, width, label='Face', color='#3498db')
    ax.bar(x, voice_scores, width, label='Voice', color='#e67e22')
    ax.bar(x + width, fused_scores, width, label='Fused', color='#2ecc71')
    ax.axhline(y=0.4, color='red', linestyle='--', alpha=0.5, label='Threshold (0.4)')
    ax.set_xticks(x)
    ax.set_xticklabels(scenarios)
    ax.set_ylabel('Score')
    ax.set_title(f'Robustness: {p1} vs {p2} (same person: {person})', fontweight='bold')
    ax.legend()
    ax.set_ylim(0, 1.1)
    plt.tight_layout()
    plt.show()

    print("\n  ┌──────────────────────────────────────────────────────────┐")
    print("  │  KEY INSIGHT:                                             │")
    print("  │  Fusion RECOVERS when ONE modality fails.                │")
    print("  │  Fusion fails ONLY when BOTH fail simultaneously.        │")
    print("  │                                                           │")
    print("  │  Real-world:                                             │")
    print("  │  • Sunglasses defeat face → voice still works            │")
    print("  │  • Noisy room defeats voice → face still works           │")
    print("  │  • Aadhaar uses 3 modalities for even more robustness    │")
    print("  └──────────────────────────────────────────────────────────┘")
else:
    print("❌ Need at least 2 samples of the same person for this demo.")

---
# Step 8: Person Identification — Full Pipeline

Use `_c` samples as **query** (person at the door), `_a` and `_b` as **database** (enrolled faces+voices).

In [ ]:
# Split into database and query
database = [name for name in common_names if not name.endswith('_c')]
queries  = [name for name in common_names if name.endswith('_c')]

# If no _c files, use last sample per person as query
if len(queries) == 0:
    for person in persons:
        person_samples = [n for n in common_names if get_person(n) == person]
        if len(person_samples) >= 2:
            queries.append(person_samples[-1])
            database = [n for n in common_names if n not in queries]

print("═" * 70)
print("Person Identification — Full Pipeline")
print("═" * 70)
print(f"\n  Database (enrolled): {database}")
print(f"  Queries (to identify): {queries}")

w_face, w_voice = 0.6, 0.4
total_correct = {"face": 0, "voice": 0, "fused": 0}
total_queries = 0

for query_name in queries:
    query_person = get_person(query_name)
    total_queries += 1

    print(f"\n  ┌─ Query: {query_name} (actual: {query_person})")
    print(f"  │ {'Database':15s} {'Face':>8s} {'Voice':>8s} {'Fused':>8s}")
    print(f"  │ {'─'*45}")

    best_face = (-1, "")
    best_voice = (-1, "")
    best_fused = (-1, "")

    for db_name in database:
        db_person = get_person(db_name)
        f_score = cosine_sim(face_data[query_name], face_data[db_name])
        v_score = cosine_sim(voice_data[query_name], voice_data[db_name])
        fused = w_face * f_score + w_voice * v_score

        if f_score > best_face[0]: best_face = (f_score, db_name)
        if v_score > best_voice[0]: best_voice = (v_score, db_name)
        if fused > best_fused[0]: best_fused = (fused, db_name)

        same = " ✅" if db_person == query_person else ""
        print(f"  │ {db_name:15s} {f_score:8.4f} {v_score:8.4f} {fused:8.4f}{same}")

    # Results
    face_id = get_person(best_face[1])
    voice_id = get_person(best_voice[1])
    fused_id = get_person(best_fused[1])

    face_ok = "✅" if face_id == query_person else "❌"
    voice_ok = "✅" if voice_id == query_person else "❌"
    fused_ok = "✅" if fused_id == query_person else "❌"

    if face_id == query_person: total_correct["face"] += 1
    if voice_id == query_person: total_correct["voice"] += 1
    if fused_id == query_person: total_correct["fused"] += 1

    print(f"  │")
    print(f"  │ Face says:  {face_id:10s} {face_ok}")
    print(f"  │ Voice says: {voice_id:10s} {voice_ok}")
    print(f"  │ Fused says: {fused_id:10s} {fused_ok}")
    print(f"  └─")

# Final accuracy
print(f"\n{'═' * 70}")
print(f"Overall Accuracy ({total_queries} queries)")
print(f"{'═' * 70}")
print(f"  Face only:  {total_correct['face']}/{total_queries} ({total_correct['face']/total_queries:.0%})")
print(f"  Voice only: {total_correct['voice']}/{total_queries} ({total_correct['voice']/total_queries:.0%})")
print(f"  FUSED:      {total_correct['fused']}/{total_queries} ({total_correct['fused']/total_queries:.0%})")

---
---

## Summary

### What We Did
- **Face embeddings** (ArcFace, 512-dim) — same concept as CLIP image embeddings
- **Voice embeddings** (Resemblyzer, 256-dim) — same concept, different modality
- **Cosine similarity** for both — same math as CLIP
- **Late fusion**: weighted combination of face score + voice score

### Key Insights
- Fusion is more **robust** than any single modality
- When face fails (sunglasses) → voice compensates
- When voice fails (noise) → face compensates
- **Weight tuning** matters — optimal ratio depends on data quality

### Connection to Unit 4
```
Late Fusion          → weighted score combination (face + voice)
Model-Free Approach  → weighted average (no learned fusion layer)
Face Recognition     → ArcFace (same embedding+similarity concept as CLIP)
Biometric System     → multimodal identity verification
```

### Real-World Systems Using This
```
Aadhaar (India):     face + iris + fingerprint
Apple Face ID:       2D RGB + 3D infrared depth
Airport e-gates:     face + passport chip
Bank authentication: voice + face (phone banking)
```

---
*Multimodal Machine Learning (21AIE541T) — Unit 4*